# LDA — Template para Novo Corpus

Copie este notebook para `notebooks/lda/02_lda_<corpus>.ipynb` e substitua os
placeholders marcados com `# <<` para adaptar a um novo corpus.

## Pipeline (5 etapas)
1. **Lematização** — spaCy (`pt_core_news_lg` ou `en_core_web_sm`), filtra POS informativas
2. **Vocabulário** — `gensim.Dictionary`, `no_below=5`, `no_above=0.5` → BoW
3. **Seleção de K** — grid search `K ∈ [k_min, k_max]`, maximiza coerência **C_v**
4. **Modelo final** — `LdaMulticore` com `K*` e 20 passes
5. **Atribuição** — `argmax(θ)` por documento → `lda_results.csv`

## Saídas
- `lda_results.csv` — `post_id, topic_id, topic_name, topic_prob_distribution`
- `lda_topics_for_eval.csv` — `topic_id, topic_name, keywords` (comparativo cross-model)
- `lda_coherence_curve.png`, `lda_wordclouds.png`, `lda_pyldavis.html`


In [ ]:
# Descomente se necessário
# !pip install gensim pyLDAvis spacy wordcloud -q
# !python -m spacy download pt_core_news_lg -q
# !python -m spacy download en_core_web_sm -q


In [ ]:
import yaml, json, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import spacy
from gensim import corpora
from gensim.models import LdaMulticore, CoherenceModel
import pyLDAvis
import pyLDAvis.gensim_models

sys.path.insert(0, str(Path(".").resolve()))
from _helpers import (
    load_params, get_corpus_config, load_corpus, get_seed,
    name_all_topics, compute_doc_distributions, export_results,
)

# ── CONFIGURAÇÃO — edite aqui para cada corpus ──────────────────────────
CORPUS_ID = "social"           # << nome do corpus no params.yaml
LANG      = "pt"               # << "pt" (PT-BR) ou "en" (EN)
# ────────────────────────────────────────────────────────────────────────

params           = load_params()
SEED, corpus_cfg = get_corpus_config(params, CORPUS_ID)
np.random.seed(SEED)

SUBDIR    = corpus_cfg.get("subdir", CORPUS_ID)
TEXT_COL  = corpus_cfg["text_column"]
ID_COL    = corpus_cfg.get("post_id_column", "post_id")

INPUT_DIR  = Path("../data/input")  / SUBDIR
OUTPUT_DIR = Path("../data/output") / SUBDIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPACY_MODEL = "pt_core_news_lg" if LANG == "pt" else "en_core_web_sm"  # << ajustar

lda_params   = params["lda"]
k_min, k_max = lda_params["k_range"]

print(f"Corpus  : {CORPUS_ID}  |  Idioma: {LANG}")
print(f"spaCy   : {SPACY_MODEL}")
print(f"K range : [{k_min}, {k_max}]")


In [ ]:
corpus_csv = INPUT_DIR / "corpus_limpo.csv"
if not corpus_csv.exists():
    raise FileNotFoundError(
        f"corpus_limpo.csv nao encontrado: {corpus_csv}\n"
        f"Execute 01-preprocessing e copie o CSV para {INPUT_DIR}"
    )

df       = load_corpus(str(INPUT_DIR))
docs     = df[TEXT_COL].astype(str).tolist()
post_ids = df[ID_COL].astype(str).tolist() if ID_COL in df.columns \
           else [f"doc_{i:04d}" for i in range(len(df))]

print(f"Docs carregados: {len(docs)}")
print(f"Exemplo: {docs[0][:120]}")


## Lematizacao (spaCy)

Filtra POS informativas: `NOUN`, `VERB`, `ADJ`.
Ajuste `POS_KEEP` se o corpus precisar de outros POS (ex.: `PROPN` para entidades nomeadas).


In [ ]:
nlp = spacy.load(SPACY_MODEL, disable=["parser", "ner"])
print(f"Modelo spaCy carregado: {SPACY_MODEL}")

POS_KEEP = {"NOUN", "VERB", "ADJ"}  # << ajustar se necessario

def lemmatize_doc(text: str) -> list:
    doc = nlp(text[:5000])
    return [
        t.lemma_.lower() for t in doc
        if t.pos_ in POS_KEEP
        and not t.is_stop
        and not t.is_punct
        and len(t.lemma_) > 2
    ]

print("Lematizando...")
t0 = time.time()
tokenized = [lemmatize_doc(d) for d in docs]
print(f"  concluido em {time.time()-t0:.1f}s")
print(f"  Exemplo: {tokenized[0][:20]}")


In [ ]:
dictionary = corpora.Dictionary(tokenized)
print(f"Vocabulario antes: {len(dictionary)} tokens")

dictionary.filter_extremes(
    no_below=lda_params["no_below"],
    no_above=lda_params["no_above"],
)
print(f"Vocabulario apos filter_extremes: {len(dictionary)} tokens")

corpus_bow = [dictionary.doc2bow(d) for d in tokenized]
non_empty  = sum(1 for d in corpus_bow if len(d) > 0)
print(f"Docs com BoW nao-vazio: {non_empty}/{len(corpus_bow)}")


## Selecao de K (grid search C_v)

Para cada `K ∈ [k_min, k_max]` treina `LdaMulticore` (10 passes) e calcula a coerencia **C_v**.
**Atencao:** C_v sistematicamente penaliza K alto (Stevens et al. 2012).
Use o K* automatico como ponto de partida, mas valide com inspecao qualitativa.


In [ ]:
scores = {}
print(f"Grid search K de {k_min} a {k_max}...")

for k in range(k_min, k_max + 1):
    m = LdaMulticore(
        corpus=corpus_bow, id2word=dictionary,
        num_topics=k, random_state=SEED, passes=10, workers=2,
    )
    cm = CoherenceModel(
        model=m, texts=tokenized,
        dictionary=dictionary, coherence="c_v",
    )
    scores[k] = cm.get_coherence()
    print(f"  K={k:3d}  c_v={scores[k]:.4f}")

best_k = max(scores, key=scores.get)
print(f"\nK* automatico (max c_v): {best_k}  (c_v={scores[best_k]:.4f})")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ks = sorted(scores)
ax.plot(ks, [scores[k] for k in ks], marker="o", linewidth=2, color="steelblue")
ax.axvline(best_k, color="red", linestyle="--", label=f"K*={best_k}")
ax.set_xlabel("K")
ax.set_ylabel("Coerencia C_v")
ax.set_title(f"Grid search K — {CORPUS_ID}")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lda_coherence_curve.png", dpi=150)
plt.show()


## Modelo final

Ajuste `BEST_K` se a inspecao qualitativa sugerir um K diferente do automatico.


In [ ]:
BEST_K = best_k  # << altere se necessario apos inspecao qualitativa

print(f"Treinando LDA final com K={BEST_K}, 20 passes...")
t0 = time.time()
model = LdaMulticore(
    corpus=corpus_bow, id2word=dictionary,
    num_topics=BEST_K, random_state=SEED,
    passes=20, workers=2,
)
print(f"  concluido em {time.time()-t0:.1f}s")

topics_keywords = {
    tid: [w for w, _ in model.show_topic(tid, topn=20)]
    for tid in range(BEST_K)
}
for tid, kws in topics_keywords.items():
    print(f"  T{tid:02d}: {', '.join(kws[:8])}")


In [ ]:
try:
    from wordcloud import WordCloud
    ncols = 3
    nrows = (BEST_K + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows))
    axes = axes.flatten()
    for i in range(BEST_K):
        ws = dict(model.show_topic(i, topn=30))
        wc = WordCloud(width=400, height=200, background_color="white",
                       max_words=25, colormap="Blues").generate_from_frequencies(ws)
        axes[i].imshow(wc, interpolation="bilinear")
        axes[i].axis("off")
        axes[i].set_title(f"T{i}", fontsize=10)
    for j in range(BEST_K, len(axes)):
        axes[j].axis("off")
    plt.suptitle(f"Wordclouds por topico — {CORPUS_ID}  (K={BEST_K})", y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "lda_wordclouds.png", dpi=150, bbox_inches="tight")
    plt.show()
except ImportError:
    print("wordcloud nao instalado — pip install wordcloud")


In [ ]:
llm_cfg      = params.get("llm", {})
topics_names = name_all_topics(
    topics_keywords,
    model=llm_cfg.get("model", "gemma4:e4b"),
    base_url=llm_cfg.get("base_url", "http://localhost:11434"),
)
print("Rotulos LLM:")
for tid, name in topics_names.items():
    print(f"  T{tid:02d}: {name}")


In [ ]:
dominant, full = compute_doc_distributions(model, corpus_bow)

counts = pd.Series(dominant).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(counts.index, counts.values, color="steelblue")
ax.set_xlabel("Topico")
ax.set_ylabel("Documentos")
ax.set_title(f"Distribuicao docs por topico — {CORPUS_ID}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lda_distribution.png", dpi=150)
plt.show()

cv_final = CoherenceModel(
    model=model, texts=tokenized,
    dictionary=dictionary, coherence="c_v",
).get_coherence()
print(f"C_v final (K={BEST_K}): {cv_final:.4f}")


In [ ]:
try:
    vis = pyLDAvis.gensim_models.prepare(model, corpus_bow, dictionary)
    pyLDAvis.save_html(vis, str(OUTPUT_DIR / "lda_pyldavis.html"))
    print(f"pyLDAvis salvo em {OUTPUT_DIR / 'lda_pyldavis.html'}")
    pyLDAvis.display(vis)
except Exception as e:
    print(f"pyLDAvis indisponivel: {e}")


In [ ]:
export_results(
    topics=dominant,
    probs=full,
    names=topics_names,
    texts=docs,
    output_dir=str(OUTPUT_DIR),
    id_col=ID_COL,
    ids=post_ids,
    model_type="lda",
    df=df,
)
print(f"Exportado: {OUTPUT_DIR}/lda_results.csv")
